In [ ]:
import difflib
import pandas as pd
import re
from nameparser import HumanName
from urllib.parse import urlparse

In [ ]:
faculty = pd.read_csv("../data/faculty_list.csv")

In [ ]:
# Check faculties without url
print(faculty[pd.isna(faculty["url"])].to_string())

In [ ]:
# Check for strange urls
def longest_common_substring_len(str1, str2):
    seq_matcher = difflib.SequenceMatcher(None, str1, str2)
    match = seq_matcher.find_longest_match(0, len(str1), 0, len(str2))
    return match.size


def is_invalid_url(row):
    url = row["url"]
    if pd.isna(url):
        return False
    url = url.lower()
    if url.endswith("/"):
        url = url[:-1]
    name = row["name"].lower()
    name_abbr = HumanName(name).first[0] + HumanName(name).last
    return (
        re.search(r"\d+(/|-)?$", url) is None  # Valid if ends with numbers
        and all(
            t.lower().replace("'", "") not in url for t in re.split(r" |-", name)
        )  # Either first or last should be in the url
        and longest_common_substring_len(
            name_abbr, urlparse(url).path + urlparse(url).query
        )
        < 4  # Name and url should have at least 4 consecutive letters in common
    )


weird_url = faculty[faculty.apply(is_invalid_url, axis=1)]
print(weird_url.to_string())